# VTA 2025 Ridership Dashboard — CPU Version (Plotly Dash)

## Story: *"Optimizing VTA Transit: Where, When & How Silicon Valley Rides"*

**Narrative flow for presentation:**
1. **Where** are the busiest stops, routes & cities?
2. **When** do people ride — peak hours, weekday vs weekend?
3. **How** are different route types performing — Frequent vs Local vs Express?
4. **Insight** — Which routes are overcrowded vs underutilized?

---
**Steps:** Data Cleaning → Build Interactive Plotly Dash Dashboard (CPU)

In [ ]:
# Install dependencies (run once)
# !pip install pandas openpyxl plotly dash dash-bootstrap-components jupyter-dash numpy

## Step 1: Load & Inspect Raw Data

In [ ]:
import pandas as pd
import numpy as np
import time

# --- Load raw data ---
start = time.time()
df_raw = pd.read_excel("OCT_2025_RBS_FULL_DATA_SET.XLSX", engine="openpyxl")
print(f"Loaded in {time.time() - start:.1f}s")
print(f"Shape: {df_raw.shape}")
print(f"\nColumns:\n{df_raw.columns.tolist()}")
df_raw.head(3)

In [ ]:
# Quick look at data types and missing values
print("--- Data Types ---")
print(df_raw.dtypes)
print("\n--- Missing Values ---")
missing = df_raw.isnull().sum()
print(missing[missing > 0])
print(f"\n--- Unique SERVICE_PERIOD values ---\n{df_raw['SERVICE_PERIOD'].unique()}")
print(f"\n--- Unique SERVICE_CODE values ---\n{df_raw['SERVICE_CODE'].unique()}")
print(f"\n--- Unique DIRECTION_NAME values ---\n{df_raw['DIRECTION_NAME'].unique()}")

## Step 2: Data Cleaning & Preprocessing

In [ ]:
start_clean = time.time()
df = df_raw.copy()

# --- 2a. Drop fully empty rows & columns ---
before = df.shape
df = df.dropna(how="all")
df = df.dropna(axis=1, how="all")
print(f"2a. Dropped all-NaN rows/cols: {before} -> {df.shape}")

# --- 2b. Fix numeric columns (coerce errors to NaN) ---
numeric_cols = [
    "BOARDINGS", "ALIGHTINGS", "TRIPS",
    "AVG_BOARDINGS", "AVG_ALIGHTINGS", "AVG_ACTIVITY",
    "PASS_LOAD", "PEAK_LOAD", "AVG_PEAK_LOAD",
    "SORT_ORDER", "STOP_ID", "TOTAL_SORT", "SORT_SP",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
print("2b. Numeric columns coerced.")

# --- 2c. Clean string columns ---
str_cols = [
    "ROUTE_NAME", "ROUTE_NUMBER", "SERVICE_PERIOD", "SERVICE_CODE",
    "DIRECTION_NAME", "BRANCH", "MAIN_CROSS_STREET", "CITY",
    "STOP_DISPLAY", "Additional_Notes", "PATTERN_KEY", "BLOCK",
]
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})
print("2c. String columns cleaned.")

# --- 2d. Convert TRIP_TIME (Excel fraction of day) to hour ---
import datetime
def trip_time_to_hour(val):
    """Convert datetime.time or numeric fraction to hour (0-23)."""
    if isinstance(val, datetime.time):
        return val.hour
    try:
        numeric = float(val)
        return int(round(numeric * 24)) % 24
    except (TypeError, ValueError):
        return np.nan

df["TRIP_HOUR"] = df["TRIP_TIME"].apply(trip_time_to_hour).astype("Int64")
print("2d. TRIP_TIME -> TRIP_HOUR extracted.")

# --- 2e. Fill missing values ---
df["CITY"] = df["CITY"].fillna("Unknown")
df["Additional_Notes"] = df["Additional_Notes"].fillna("None")
for col in ["BOARDINGS", "ALIGHTINGS", "AVG_BOARDINGS", "AVG_ALIGHTINGS", "AVG_ACTIVITY"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)
print("2e. Missing values filled (CITY->Unknown, ridership->0).")

# --- 2f. Derive: ROUTE_TYPE ---
def classify_route(sc):
    if pd.isna(sc):
        return "Unknown"
    sc = str(sc).lower()
    if "express" in sc:
        return "Express"
    elif "frequent" in sc:
        return "Frequent"
    elif "local" in sc:
        return "Local"
    elif "light rail" in sc or "lrt" in sc:
        return "Light Rail"
    elif "rapid" in sc:
        return "Rapid"
    return "Other"

df["ROUTE_TYPE"] = df["SERVICE_CODE"].apply(classify_route)
print(f"2f. ROUTE_TYPE distribution:\n{df['ROUTE_TYPE'].value_counts()}")

# --- 2g. Derive: TIME_PERIOD ---
def categorize_time(h):
    if pd.isna(h):
        return "Unknown"
    h = int(h)
    if 5 <= h < 9:
        return "AM Peak (5-9)"
    elif 9 <= h < 15:
        return "Midday (9-3)"
    elif 15 <= h < 19:
        return "PM Peak (3-7)"
    elif 19 <= h < 23:
        return "Evening (7-11)"
    return "Late Night (11-5)"

df["TIME_PERIOD"] = df["TRIP_HOUR"].apply(categorize_time)
print(f"2g. TIME_PERIOD distribution:\n{df['TIME_PERIOD'].value_counts()}")

# --- 2h. Derive: TOTAL_ACTIVITY ---
df["TOTAL_ACTIVITY"] = df["BOARDINGS"] + df["ALIGHTINGS"]
print("2h. TOTAL_ACTIVITY column created.")

# --- 2i. Drop duplicates ---
dupes = df.duplicated().sum()
if dupes > 0:
    df = df.drop_duplicates()
    print(f"2i. Removed {dupes} duplicate rows.")
else:
    print(f"2i. No duplicate rows found.")

clean_time = time.time() - start_clean
print(f"\n=== Cleaning complete in {clean_time:.2f}s | Final shape: {df.shape} ===")
df.head()

## Step 3: Pre-aggregate Data for Dashboard Performance

We pre-compute aggregated DataFrames so the dashboard callbacks are fast.

In [ ]:
start_agg = time.time()

# --- Agg 1: Top 20 busiest stops (total boardings) ---
agg_stops = (
    df.groupby(["STOP_ID", "MAIN_CROSS_STREET", "CITY"], observed=True)
    .agg(Total_Boardings=("BOARDINGS", "sum"),
         Total_Alightings=("ALIGHTINGS", "sum"),
         Total_Activity=("TOTAL_ACTIVITY", "sum"),
         Avg_Peak_Load=("PEAK_LOAD", "mean"))
    .reset_index()
    .sort_values("Total_Boardings", ascending=False)
)
top_stops = agg_stops.head(20)
print(f"Top 20 stops by boardings computed. Top stop: {top_stops.iloc[0]['MAIN_CROSS_STREET']}")

# --- Agg 2: Ridership by City ---
agg_city = (
    df.groupby("CITY", observed=True)
    .agg(Total_Boardings=("BOARDINGS", "sum"),
         Total_Alightings=("ALIGHTINGS", "sum"),
         Num_Stops=("STOP_ID", "nunique"),
         Num_Routes=("ROUTE_NUMBER", "nunique"))
    .reset_index()
    .sort_values("Total_Boardings", ascending=False)
)
print(f"City-level aggregation: {len(agg_city)} cities")

# --- Agg 3: Ridership by Route ---
agg_route = (
    df.groupby(["ROUTE_NUMBER", "ROUTE_NAME", "ROUTE_TYPE"], observed=True)
    .agg(Total_Boardings=("BOARDINGS", "sum"),
         Total_Alightings=("ALIGHTINGS", "sum"),
         Avg_Peak_Load=("PEAK_LOAD", "mean"),
         Avg_Pass_Load=("PASS_LOAD", "mean"),
         Num_Trips=("TRIPS", "sum"))
    .reset_index()
    .sort_values("Total_Boardings", ascending=False)
)
print(f"Route-level aggregation: {len(agg_route)} routes")

# --- Agg 4: Hourly ridership pattern (heatmap data) ---
agg_hourly = (
    df.groupby(["TRIP_HOUR", "SERVICE_PERIOD"], observed=True)
    .agg(Total_Boardings=("BOARDINGS", "sum"))
    .reset_index()
)
print(f"Hourly x ServicePeriod aggregation: {len(agg_hourly)} rows")

# --- Agg 5: Route type comparison ---
agg_route_type = (
    df.groupby("ROUTE_TYPE", observed=True)
    .agg(Total_Boardings=("BOARDINGS", "sum"),
         Avg_Boardings=("AVG_BOARDINGS", "mean"),
         Avg_Peak_Load=("PEAK_LOAD", "mean"),
         Num_Routes=("ROUTE_NUMBER", "nunique"),
         Num_Stops=("STOP_ID", "nunique"))
    .reset_index()
    .sort_values("Total_Boardings", ascending=False)
)
print(f"Route type aggregation: {len(agg_route_type)} types")

# --- Agg 6: Time period x Route type (for stacked analysis) ---
agg_time_route = (
    df.groupby(["TIME_PERIOD", "ROUTE_TYPE"], observed=True)
    .agg(Total_Boardings=("BOARDINGS", "sum"))
    .reset_index()
)
print(f"TimePeriod x RouteType: {len(agg_time_route)} rows")

# --- Agg 7: Weekday vs Weekend by Route ---
agg_service = (
    df.groupby(["ROUTE_NAME", "SERVICE_PERIOD"], observed=True)
    .agg(Total_Boardings=("BOARDINGS", "sum"),
         Avg_Peak_Load=("PEAK_LOAD", "mean"))
    .reset_index()
)
print(f"Route x ServicePeriod: {len(agg_service)} rows")

# --- Agg 8: Direction-level ridership ---
agg_direction = (
    df.groupby(["ROUTE_NAME", "DIRECTION_NAME"], observed=True)
    .agg(Total_Boardings=("BOARDINGS", "sum"),
         Total_Alightings=("ALIGHTINGS", "sum"))
    .reset_index()
)
print(f"Route x Direction: {len(agg_direction)} rows")

agg_time = time.time() - start_agg
print(f"\n=== All aggregations done in {agg_time:.2f}s (CPU) ===")

## Step 4: Build Interactive Plotly Dash Dashboard

**Story Flow — 9 Visuals across 3 Tabs:**
- **Tab 1 — WHERE:** (1) Top 20 busiest stops, (2) City ridership treemap, (3) Top 15 routes
- **Tab 2 — WHEN:** (4) Hourly ridership heatmap, (5) Time period by route type, (6) Weekday vs weekend
- **Tab 3 — HOW:** (7) Route type bubble chart, (8) Boarding vs alighting scatter, (9) Top 20 overcrowded routes

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc

# ============================================================
# COLOR PALETTE & CONSTANTS
# ============================================================
COLORS = {
    "Frequent": "#1f77b4",
    "Local": "#ff7f0e",
    "Express": "#2ca02c",
    "Light Rail": "#d62728",
    "Rapid": "#9467bd",
    "Other": "#8c564b",
    "Unknown": "#7f7f7f",
}

TIME_ORDER = ["Late Night (11-5)", "AM Peak (5-9)", "Midday (9-3)",
              "PM Peak (3-7)", "Evening (7-11)"]

ALL_ROUTE_TYPES = sorted(df["ROUTE_TYPE"].unique().tolist())
ALL_SERVICE_PERIODS = sorted(df["SERVICE_PERIOD"].dropna().unique().tolist())

# ============================================================
# HELPER FUNCTIONS — Build each figure from filtered data
# ============================================================

def build_fig_top_stops(filtered, title_suffix="All Cities"):
    """Fig 1: Top 20 Busiest Stops (horizontal bar)"""
    stops_f = (
        filtered.groupby(["STOP_ID", "MAIN_CROSS_STREET", "CITY"], observed=True)
        .agg(Total_Boardings=("BOARDINGS", "sum"),
             Total_Alightings=("ALIGHTINGS", "sum"),
             Avg_Peak_Load=("PEAK_LOAD", "mean"))
        .reset_index()
        .sort_values("Total_Boardings", ascending=False)
        .head(20)
    )
    fig = px.bar(
        stops_f.sort_values("Total_Boardings"),
        x="Total_Boardings", y="MAIN_CROSS_STREET",
        orientation="h", color="CITY",
        title=f"Fig 1: Top 20 Busiest Stops — {title_suffix}",
        labels={"Total_Boardings": "Total Boardings", "MAIN_CROSS_STREET": "Stop"},
        hover_data=["CITY", "Total_Alightings", "Avg_Peak_Load"],
    )
    fig.update_layout(yaxis=dict(tickfont=dict(size=10)), height=550)
    return fig


def build_fig_city(filtered):
    """Fig 2: City-level Ridership (treemap)"""
    agg = (
        filtered.groupby("CITY", observed=True)
        .agg(Total_Boardings=("BOARDINGS", "sum"),
             Num_Stops=("STOP_ID", "nunique"),
             Num_Routes=("ROUTE_NUMBER", "nunique"))
        .reset_index()
        .sort_values("Total_Boardings", ascending=False)
        .head(15)
    )
    fig = px.treemap(
        agg, path=["CITY"], values="Total_Boardings",
        color="Total_Boardings", color_continuous_scale="Blues",
        title="Fig 2: Ridership Distribution by City (Top 15)",
        hover_data=["Num_Stops", "Num_Routes"],
    )
    return fig


def build_fig_routes(filtered, title_suffix="All Cities"):
    """Fig 3: Top 15 Routes by Boardings"""
    routes_f = (
        filtered.groupby(["ROUTE_NUMBER", "ROUTE_NAME", "ROUTE_TYPE"], observed=True)
        .agg(Total_Boardings=("BOARDINGS", "sum"),
             Avg_Peak_Load=("PEAK_LOAD", "mean"),
             Num_Trips=("TRIPS", "sum"))
        .reset_index()
        .sort_values("Total_Boardings", ascending=False)
        .head(15)
    )
    fig = px.bar(
        routes_f.sort_values("Total_Boardings"),
        x="Total_Boardings", y="ROUTE_NAME",
        orientation="h", color="ROUTE_TYPE", color_discrete_map=COLORS,
        title=f"Fig 3: Top 15 Routes — {title_suffix}",
        labels={"Total_Boardings": "Total Boardings", "ROUTE_NAME": "Route"},
        hover_data=["Avg_Peak_Load", "Num_Trips"],
    )
    fig.update_layout(yaxis=dict(tickfont=dict(size=9)), height=500)
    return fig


def build_fig_heatmap(filtered):
    """Fig 4: Hourly Ridership Heatmap"""
    agg = (
        filtered.groupby(["TRIP_HOUR", "SERVICE_PERIOD"], observed=True)
        .agg(Total_Boardings=("BOARDINGS", "sum"))
        .reset_index()
    )
    pivot = agg.pivot_table(
        index="SERVICE_PERIOD", columns="TRIP_HOUR",
        values="Total_Boardings", fill_value=0
    )
    fig = px.imshow(
        pivot,
        labels=dict(x="Hour of Day", y="Service Period", color="Total Boardings"),
        title="Fig 4: Ridership Heatmap — Hour of Day vs Service Period",
        color_continuous_scale="YlOrRd", aspect="auto",
    )
    fig.update_layout(height=350)
    return fig


def build_fig_time_route(filtered):
    """Fig 5: Boardings by Time Period & Route Type"""
    agg = (
        filtered.groupby(["TIME_PERIOD", "ROUTE_TYPE"], observed=True)
        .agg(Total_Boardings=("BOARDINGS", "sum"))
        .reset_index()
    )
    fig = px.bar(
        agg, x="TIME_PERIOD", y="Total_Boardings",
        color="ROUTE_TYPE", color_discrete_map=COLORS, barmode="group",
        title="Fig 5: Boardings by Time Period & Route Type",
        category_orders={"TIME_PERIOD": TIME_ORDER},
    )
    fig.update_layout(height=400)
    return fig


def build_fig_weekday(filtered):
    """Fig 6: Weekday vs Weekend — Top 10 Routes"""
    agg_route_f = (
        filtered.groupby(["ROUTE_NAME"], observed=True)
        .agg(Total_Boardings=("BOARDINGS", "sum"))
        .reset_index()
        .sort_values("Total_Boardings", ascending=False)
    )
    top10 = agg_route_f.head(10)["ROUTE_NAME"].tolist()
    agg_svc = (
        filtered[filtered["ROUTE_NAME"].isin(top10)]
        .groupby(["ROUTE_NAME", "SERVICE_PERIOD"], observed=True)
        .agg(Total_Boardings=("BOARDINGS", "sum"))
        .reset_index()
    )
    fig = px.bar(
        agg_svc, x="ROUTE_NAME", y="Total_Boardings",
        color="SERVICE_PERIOD", barmode="group",
        title="Fig 6: Weekday vs Weekend Ridership — Top 10 Routes",
        labels={"ROUTE_NAME": "Route", "Total_Boardings": "Total Boardings"},
    )
    fig.update_layout(xaxis_tickangle=-45, height=450)
    return fig


def build_fig_bubble(filtered):
    """Fig 7: Route Type Performance (bubble chart)"""
    agg = (
        filtered.groupby("ROUTE_TYPE", observed=True)
        .agg(Total_Boardings=("BOARDINGS", "sum"),
             Avg_Peak_Load=("PEAK_LOAD", "mean"),
             Num_Routes=("ROUTE_NUMBER", "nunique"),
             Num_Stops=("STOP_ID", "nunique"))
        .reset_index()
        .sort_values("Total_Boardings", ascending=False)
    )
    fig = px.scatter(
        agg, x="Num_Routes", y="Total_Boardings",
        size="Avg_Peak_Load", color="ROUTE_TYPE", color_discrete_map=COLORS,
        title="Fig 7: Route Type Performance (bubble = Avg Peak Load)",
        labels={"Num_Routes": "Number of Routes", "Total_Boardings": "Total Boardings"},
        hover_data=["Num_Stops", "Avg_Peak_Load"],
    )
    fig.update_layout(height=450)
    return fig


def build_fig_balance(filtered):
    """Fig 8: Boarding vs Alighting Balance"""
    agg = (
        filtered.groupby(["ROUTE_NUMBER", "ROUTE_NAME", "ROUTE_TYPE"], observed=True)
        .agg(Total_Boardings=("BOARDINGS", "sum"),
             Total_Alightings=("ALIGHTINGS", "sum"))
        .reset_index()
    )
    fig = px.scatter(
        agg, x="Total_Boardings", y="Total_Alightings",
        color="ROUTE_TYPE", color_discrete_map=COLORS, hover_name="ROUTE_NAME",
        title="Fig 8: Boarding vs Alighting Balance by Route",
        labels={"Total_Boardings": "Total Boardings", "Total_Alightings": "Total Alightings"},
    )
    if len(agg) > 0:
        max_val = max(agg["Total_Boardings"].max(), agg["Total_Alightings"].max())
        fig.add_trace(go.Scatter(
            x=[0, max_val], y=[0, max_val],
            mode="lines", line=dict(dash="dash", color="gray"),
            showlegend=False,
        ))
    fig.update_layout(height=450)
    return fig


def build_fig_load(filtered):
    """Fig 9: Top 20 Most Crowded Routes"""
    agg = (
        filtered.groupby(["ROUTE_NUMBER", "ROUTE_NAME", "ROUTE_TYPE"], observed=True)
        .agg(Avg_Peak_Load=("PEAK_LOAD", "mean"))
        .reset_index()
        .nlargest(20, "Avg_Peak_Load")
    )
    fig = px.bar(
        agg.sort_values("Avg_Peak_Load"),
        x="Avg_Peak_Load", y="ROUTE_NAME",
        orientation="h", color="ROUTE_TYPE", color_discrete_map=COLORS,
        title="Fig 9: Top 20 Most Crowded Routes (Avg Peak Load)",
        labels={"Avg_Peak_Load": "Avg Peak Load (passengers)", "ROUTE_NAME": "Route"},
    )
    fig.update_layout(yaxis=dict(tickfont=dict(size=9)), height=550)
    return fig


print("All 9 figure-builder functions defined.")

## Step 5: Launch the Dash Dashboard

**Interactive Components (3):**
1. **City Dropdown** — filters Tab 1 visuals + updates KPIs
2. **Route Type Checklist** — filters all 3 tabs + updates KPIs
3. **Service Period Checklist** — filters Tab 2 visuals

**Callbacks (4):**
1. **KPI Callback** — dynamically updates all 4 KPI cards based on City + Route Type filters
2. **Tab 1 Callback** — updates Fig 1, 2, 3 based on City + Route Type
3. **Tab 2 Callback** — updates Fig 4, 5, 6 based on Route Type + Service Period
4. **Tab 3 Callback** — updates Fig 7, 8, 9 based on City + Route Type

In [ ]:
# ============================================================
# DASH APP — 3 Interactive Components, 4 Callbacks, 9 Visuals
# ============================================================

app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])

# --- Filter Options ---
city_options = [{"label": "All Cities", "value": "ALL"}] + [
    {"label": c, "value": c}
    for c in sorted(df["CITY"].dropna().unique()) if c != "Unknown"
]

# --- Global Filters Bar (above tabs) ---
filters_bar = dbc.Card(
    dbc.CardBody([
        dbc.Row([
            dbc.Col([
                html.Label("Filter by City:", className="fw-bold"),
                dcc.Dropdown(
                    id="city-filter", options=city_options,
                    value="ALL", clearable=False,
                ),
            ], md=3),
            dbc.Col([
                html.Label("Filter by Route Type:", className="fw-bold"),
                dcc.Checklist(
                    id="route-type-filter",
                    options=[{"label": f"  {rt}", "value": rt} for rt in ALL_ROUTE_TYPES],
                    value=ALL_ROUTE_TYPES,
                    inline=True,
                    inputStyle={"marginRight": "4px", "marginLeft": "12px"},
                ),
            ], md=9),
        ]),
    ]),
    className="mb-3 shadow-sm",
)

# --- KPI Cards (dynamic — updated by Callback 1) ---
kpi_row = dbc.Row([
    dbc.Col(dbc.Card(dbc.CardBody([
        html.H6("Total Boardings", className="card-title text-muted", style={"fontSize": "0.85rem"}),
        html.H4(id="kpi-boardings", className="text-primary fw-bold"),
    ]), className="shadow-sm"), md=3),
    dbc.Col(dbc.Card(dbc.CardBody([
        html.H6("Unique Stops", className="card-title text-muted", style={"fontSize": "0.85rem"}),
        html.H4(id="kpi-stops", className="text-success fw-bold"),
    ]), className="shadow-sm"), md=3),
    dbc.Col(dbc.Card(dbc.CardBody([
        html.H6("Routes Served", className="card-title text-muted", style={"fontSize": "0.85rem"}),
        html.H4(id="kpi-routes", className="text-info fw-bold"),
    ]), className="shadow-sm"), md=3),
    dbc.Col(dbc.Card(dbc.CardBody([
        html.H6("Cities Covered", className="card-title text-muted", style={"fontSize": "0.85rem"}),
        html.H4(id="kpi-cities", className="text-warning fw-bold"),
    ]), className="shadow-sm"), md=3),
], className="mb-3")

# --- Tab 1: WHERE ---
tab1 = dbc.Tab(label="WHERE: Stops & Cities", children=[
    dbc.Container([
        html.Br(),
        html.P("Which stops, routes, and cities generate the most ridership?",
               className="text-muted fst-italic"),
        dbc.Row([
            dbc.Col(dcc.Graph(id="fig-top-stops"), md=7),
            dbc.Col(dcc.Graph(id="fig-city"), md=5),
        ]),
        dbc.Row([
            dbc.Col(dcc.Graph(id="fig-routes"), md=12),
        ]),
    ], fluid=True),
])

# --- Tab 2: WHEN (has its own Service Period filter) ---
tab2 = dbc.Tab(label="WHEN: Time Patterns", children=[
    dbc.Container([
        html.Br(),
        html.P("When do riders use VTA the most — by hour, time period, and day type?",
               className="text-muted fst-italic"),
        dbc.Row([
            dbc.Col([
                html.Label("Filter by Service Period:", className="fw-bold"),
                dcc.Checklist(
                    id="service-period-filter",
                    options=[{"label": f"  {sp}", "value": sp} for sp in ALL_SERVICE_PERIODS],
                    value=ALL_SERVICE_PERIODS,
                    inline=True,
                    inputStyle={"marginRight": "4px", "marginLeft": "12px"},
                ),
            ], md=12),
        ], className="mb-3"),
        dbc.Row([
            dbc.Col(dcc.Graph(id="fig-heatmap"), md=12),
        ]),
        dbc.Row([
            dbc.Col(dcc.Graph(id="fig-time-route"), md=6),
            dbc.Col(dcc.Graph(id="fig-weekday"), md=6),
        ]),
    ], fluid=True),
])

# --- Tab 3: HOW ---
tab3 = dbc.Tab(label="HOW: Performance & Crowding", children=[
    dbc.Container([
        html.Br(),
        html.P("How do different route types perform, and where is overcrowding or underutilization?",
               className="text-muted fst-italic"),
        dbc.Row([
            dbc.Col(dcc.Graph(id="fig-bubble"), md=6),
            dbc.Col(dcc.Graph(id="fig-balance"), md=6),
        ]),
        dbc.Row([
            dbc.Col(dcc.Graph(id="fig-load"), md=12),
        ]),
    ], fluid=True),
])

# --- Full Layout ---
app.layout = dbc.Container([
    html.H2("VTA 2025 Ridership Dashboard", className="text-center mt-3 mb-2 fw-bold"),
    html.P("Optimizing Silicon Valley Transit: Where, When & How People Ride",
           className="text-center text-muted mb-3"),
    filters_bar,
    kpi_row,
    html.Hr(),
    dbc.Tabs([tab1, tab2, tab3]),
    html.Hr(),
    html.Footer(
        html.Small("Data: VTA Oct 2025 Ridership by Stop & Station | Dashboard: Plotly Dash (CPU)",
                   className="text-muted"),
        className="text-center mb-3"
    ),
], fluid=True)


# ============================================================
# HELPER: Apply global filters (City + Route Type)
# ============================================================
def apply_global_filters(city, route_types):
    filtered = df[df["ROUTE_TYPE"].isin(route_types)]
    if city != "ALL":
        filtered = filtered[filtered["CITY"] == city]
    return filtered


# ============================================================
# CALLBACK 1: KPI Cards (City + Route Type → 4 KPI text values)
# ============================================================
@app.callback(
    [Output("kpi-boardings", "children"),
     Output("kpi-stops", "children"),
     Output("kpi-routes", "children"),
     Output("kpi-cities", "children")],
    [Input("city-filter", "value"),
     Input("route-type-filter", "value")]
)
def update_kpis(city, route_types):
    filtered = apply_global_filters(city, route_types)
    boardings = f"{int(filtered['BOARDINGS'].sum()):,}"
    stops = f"{filtered['STOP_ID'].nunique():,}"
    routes = f"{filtered['ROUTE_NUMBER'].nunique():,}"
    cities = f"{filtered['CITY'].nunique():,}"
    return boardings, stops, routes, cities


# ============================================================
# CALLBACK 2: Tab 1 — WHERE (City + Route Type → Fig 1, 2, 3)
# ============================================================
@app.callback(
    [Output("fig-top-stops", "figure"),
     Output("fig-city", "figure"),
     Output("fig-routes", "figure")],
    [Input("city-filter", "value"),
     Input("route-type-filter", "value")]
)
def update_tab1(city, route_types):
    filtered = apply_global_filters(city, route_types)
    title_suffix = city if city != "ALL" else "All Cities"
    return (
        build_fig_top_stops(filtered, title_suffix),
        build_fig_city(filtered),
        build_fig_routes(filtered, title_suffix),
    )


# ============================================================
# CALLBACK 3: Tab 2 — WHEN (Route Type + Service Period → Fig 4, 5, 6)
# ============================================================
@app.callback(
    [Output("fig-heatmap", "figure"),
     Output("fig-time-route", "figure"),
     Output("fig-weekday", "figure")],
    [Input("route-type-filter", "value"),
     Input("service-period-filter", "value")]
)
def update_tab2(route_types, service_periods):
    filtered = df[
        df["ROUTE_TYPE"].isin(route_types) &
        df["SERVICE_PERIOD"].isin(service_periods)
    ]
    return (
        build_fig_heatmap(filtered),
        build_fig_time_route(filtered),
        build_fig_weekday(filtered),
    )


# ============================================================
# CALLBACK 4: Tab 3 — HOW (City + Route Type → Fig 7, 8, 9)
# ============================================================
@app.callback(
    [Output("fig-bubble", "figure"),
     Output("fig-balance", "figure"),
     Output("fig-load", "figure")],
    [Input("city-filter", "value"),
     Input("route-type-filter", "value")]
)
def update_tab3(city, route_types):
    filtered = apply_global_filters(city, route_types)
    return (
        build_fig_bubble(filtered),
        build_fig_balance(filtered),
        build_fig_load(filtered),
    )


print("Dashboard ready! 3 interactive components + 4 callbacks + 9 visuals + 4 dynamic KPIs")

In [ ]:
# ============================================================
# RUN THE DASHBOARD
# ============================================================
# Option 1: Run inline in Jupyter (uncomment the next line)
# app.run(jupyter_mode="inline", port=8050)

# Option 2: Run in a new browser tab (recommended for full experience)
app.run(debug=False, port=8050)

# After running, open http://127.0.0.1:8050 in your browser